In [ ]:
# 1. 导入核心库
import torch
from torch import nn
from d2l import torch as d2l

# 2. 定义超参数并加载数据
batch_size = 256  # 设置每个批次读取的样本数量
# 使用 d2l 库的内置函数加载 Fashion-MNIST 数据集
# train_iter 是训练集的迭代器，test_iter 是测试集的迭代器
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

# 3. 定义模型结构和参数
# 定义模型的输入、输出和隐藏层大小
num_inputs, num_outputs, num_hiddens = 784, 10, 256
# num_inputs:  输入层大小。Fashion-MNIST 图片为 28x28，展平后为 784。
# num_outputs: 输出层大小。Fashion-MNIST 有 10 个类别。
# num_hiddens: 隐藏层神经元数量。这是一个可以调整的超参数，决定了模型的复杂度。

# --- 定义第一层（输入层 -> 隐藏层）的参数 ---
W1 = nn.Parameter(
    torch.randn(num_inputs, num_hiddens, requires_grad=True) * 0.01
)
# W1: 权重矩阵，形状为 (784, 256)。
# nn.Parameter(...) 将其注册为模型的可训练参数。
# torch.randn(...) 创建服从标准正态分布的随机张量。
# * 0.01 缩小初始权重的范围，有助于稳定训练初期。

b1 = nn.Parameter(torch.zeros(num_hiddens, requires_grad=True))
# b1: 偏置向量，形状为 (256,)。初始化为全零。

# --- 定义第二层（隐藏层 -> 输出层）的参数 ---
W2 = nn.Parameter(torch.randn(
    num_hiddens, num_outputs, requires_grad=True) * 0.01
)
# W2: 权重矩阵，形状为 (256, 10)。

b2 = nn.Parameter(torch.zeros(num_outputs, requires_grad=True))
# b2: 偏置向量，形状为 (10,)。初始化为全零。

# 将所有参数放入一个列表中，方便后续传入优化器
params = [W1, b1, W2, b2]

# 4. 定义激活函数
def relu(X):
    """
    ReLU 激活函数: f(x) = max(0, x)
    作用：为模型引入非线性，使其能学习更复杂的模式。
    """
    # 创建一个与输入 X 形状相同、元素全为 0 的张量
    a = torch.zeros_like(X)
    # 返回 X 和 a 按元素比较的最大值
    return torch.max(X, a)

# 5. 定义模型网络结构（前向传播）
def net(X):
    """
    定义多层感知机的前向传播过程。
    输入: X, 形状为 (batch_size, 1, 28, 28)
    输出: 模型的预测 logits, 形状为 (batch_size, 10)
    """
    # 将输入的图片数据从 (batch_size, 1, 28, 28) 展平为 (batch_size, 784)
    X = X.reshape((-1, num_inputs))
    # 计算隐藏层的输出。`@` 是矩阵乘法运算符。
    H = relu(X @ W1 + b1)
    # 计算输出层的输出，并返回结果
    return (H @ W2 + b2)

# 6. 定义损失函数
loss = nn.CrossEntropyLoss(reduction='none')
# CrossEntropyLoss: 交叉熵损失，是分类任务中最常用的损失函数。
# 它内部会自动对模型的输出执行 Softmax 操作。
# reduction='none': 表示返回每个样本的损失，而不是对一个批次的损失求和或求平均。

# 7. 定义优化器和训练超参数
num_epochs, lr = 10, 0.1  # num_epochs: 训练总轮数; lr: 学习率
# 使用随机梯度下降（SGD）作为优化器
updater = torch.optim.SGD(params, lr=lr)

# 8. 定义并执行训练循环
def train_loop():
    """完整的训练和评估循环"""
    # 外层循环，控制训练的总轮数 (epochs)
    for epoch in range(num_epochs):
        # --- 训练阶段 ---
        # 初始化用于累计每轮训练指标的变量
        train_loss_sum, train_acc_sum, n = 0.0, 0.0, 0
        
        # 内层循环，遍历训练数据集的每个批次
        for X, y in train_iter:
            # a. 前向传播：计算模型预测值
            y_hat = net(X)
            
            # b. 计算损失：计算当前批次的平均损失
            l = loss(y_hat, y).mean()
            
            # c. 反向传播与优化
            updater.zero_grad()  # 清除上一批次计算的梯度，防止累积
            l.backward()         # 自动计算损失对所有可训练参数的梯度
            updater.step()       # 根据梯度更新模型参数 (W1, b1, W2, b2)
            
            # d. 累计指标
            train_loss_sum += l.item()  # .item() 将单元素张量转为 Python 数字
            # y_hat.argmax(dim=1) 找到每个样本预测概率最高的类别索引
            train_acc_sum += (y_hat.argmax(dim=1) == y).sum().item()
            n += y.shape[0]  # 累加已处理的样本总数

        # --- 评估阶段 ---
        # 初始化用于累计测试集指标的变量
        test_acc_sum, test_n = 0.0, 0
        # `with torch.no_grad():` 上下文管理器，临时禁用梯度计算，以加速评估过程
        with torch.no_grad():
            for X, y in test_iter:
                y_hat = net(X)
                test_acc_sum += (y_hat.argmax(dim=1) == y).sum().item()
                test_n += y.shape[0]

        # 打印当前轮次的训练和评估结果
        print(f'epoch {epoch + 1}, loss {train_loss_sum / n:.4f}, '
              f'train acc {train_acc_sum / n:.4f}, '
              f'test acc {test_acc_sum / test_n:.4f}')

# 调用函数，开始训练
train_loop()
